In [15]:
from __future__ import annotations

# =============================================================================
# CELL 1: IMPORTS
# =============================================================================

import logging
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

from src.Chunking.chunking import get_chunks
from src.Config import get_settings

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

print("✓ All imports complete")

✓ All imports complete


In [16]:
# =============================================================================
# CELL 2: FILTER HELPERS
# =============================================================================

def filter_original_only(docs: List[Document]) -> List[Document]:
    """Keep only original law articles (excludes preambles, tables, amendments)."""
    return [d for d in docs if d.metadata.get("chunk_type") == "article"]


def filter_amended_only(docs: List[Document]) -> List[Document]:
    """Keep only amended articles (from new* files)."""
    return [d for d in docs if d.metadata.get("chunk_type") == "amended_article"]


def filter_latest_versions(docs: List[Document]) -> List[Document]:
    """
    For each (law_id, article_number) pair keep only the most recent version.

    Priority order:
        1. amended_article with the latest amendment_date
        2. original article (if no amendment exists)

    Chunks with no article_number (preambles, tables) are always kept as-is.
    This gives a clean, deduplicated corpus for RAG.
    """
    # Pass-through chunks that have no article_number (preambles, tables)
    no_number: List[Document] = [
        d for d in docs if not d.metadata.get("article_number")
    ]

    # Group the rest by (law_id, article_number)
    groups: Dict[tuple, List[Document]] = defaultdict(list)
    for doc in docs:
        m          = doc.metadata
        article_no = m.get("article_number")
        if not article_no:
            continue
        key = (m.get("law_id"), article_no)
        groups[key].append(doc)

    latest: List[Document] = []
    for _, group in groups.items():
        amended = [
            d for d in group
            if d.metadata.get("chunk_type") == "amended_article"
            and d.metadata.get("amendment_date")
        ]
        if amended:
            amended.sort(key=lambda d: d.metadata["amendment_date"], reverse=True)
            latest.append(amended[0])
        else:
            originals = [
                d for d in group if d.metadata.get("chunk_type") == "article"
            ]
            latest.extend(originals[:1] if originals else group[:1])

    return no_number + latest


print("✓ Filter helpers defined")


✓ Filter helpers defined


In [17]:
# =============================================================================
# CELL 3: VECTOR STORE
# =============================================================================

class LegalVectorStore:
    """
    Wraps any LangChain vector store with legal-specific helpers.

    Supported backends: ``'faiss'``, ``'chroma'``

    Metadata keys available for filtering
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ``law_id``              e.g. ``'penal_code'``
    ``law_title``           human-readable Arabic title
    ``chunk_type``          ``'article'`` | ``'amended_article'`` |
                            ``'preamble'`` | ``'table'``
    ``article_number``      Western-digit string, e.g. ``'45'``
    ``source_file``         original filename
    ``amendment_date``      ISO date string (amended_article only)
    ``amendment_law_number``(amended_article only)
    ``amendment_type``      ``'modification'`` | ``'addition'`` | ``'deletion'``
    """

    def __init__(self, vectorstore, docs: List[Document]) -> None:
        self.vectorstore = vectorstore
        self._docs       = docs

    # ── constructors ──────────────────────────────────────────────────────

    @classmethod
    def from_documents(
        cls,
        docs:               List[Document],
        embeddings,
        backend:            str            = "faiss",
        docs_filter:        str            = "latest",
        faiss_index_path:   Optional[str]  = None,
        chroma_persist_dir: Optional[str]  = "./chroma_legal_db",
        chroma_collection:  str            = "egyptian_legal_corpus",
        verbose:            bool           = True,
    ) -> "LegalVectorStore":
        """
        Build a vector store from a list of :class:`Document` objects.

        Args:
            docs:               Output of :func:`get_chunks`.
            embeddings:         Any LangChain ``Embeddings`` object.
            backend:            ``'faiss'`` or ``'chroma'``.
            docs_filter:        ``'all'``      — index everything
                                ``'latest'``   — amended version wins over original ✓
                                ``'original'`` — original articles only
            faiss_index_path:   Save FAISS index to this path (``None`` = skip).
            chroma_persist_dir: Chroma persistence directory.
            chroma_collection:  Chroma collection name.
            verbose:            Print progress messages.
        """
        # Apply filter
        if docs_filter == "latest":
            index_docs = filter_latest_versions(docs)
        elif docs_filter == "original":
            index_docs = filter_original_only(docs)
        else:                           # 'all'
            index_docs = docs

        logger.info(
            f"  Filter='{docs_filter}' → {len(index_docs):,} docs "
            f"(from {len(docs):,})"
        )

        if not index_docs:
            raise ValueError("No documents to index after filtering.")

        if backend == "faiss":
            vs = cls._build_faiss(index_docs, embeddings, faiss_index_path, verbose)
        elif backend == "chroma":
            vs = cls._build_chroma(
                index_docs, embeddings,
                chroma_persist_dir, chroma_collection, verbose,
            )
        else:
            raise ValueError(f"Unknown backend '{backend}'. Use 'faiss' or 'chroma'.")

        return cls(vs, index_docs)

    @classmethod
    def _build_faiss(
        cls,
        docs,
        embeddings,
        index_path: Optional[str],
        verbose:    bool,
    ):
        try:
            from langchain_community.vectorstores import FAISS
        except ImportError:
            raise ImportError(
                "Run: pip install langchain-community faiss-cpu"
            )
        if verbose:
            print(f"  Building FAISS index over {len(docs):,} documents …")
        vs = FAISS.from_documents(docs, embeddings)
        if index_path:
            vs.save_local(index_path)
            logger.info(f"  ✓ FAISS index saved → {index_path}")
        if verbose:
            print("  ✓ FAISS index ready")
        return vs

    @classmethod
    def _build_chroma(
        cls,
        docs,
        embeddings,
        persist_dir: Optional[str],
        collection:  str,
        verbose:     bool,
    ):
        try:
            from langchain_community.vectorstores import Chroma
        except ImportError:
            raise ImportError(
                "Run: pip install langchain-community chromadb"
            )
        if verbose:
            print(f"  Building Chroma index over {len(docs):,} documents …")
        vs = Chroma.from_documents(
            docs, embeddings,
            collection_name   = collection,
            persist_directory = persist_dir,
        )
        if verbose:
            print(f"  ✓ Chroma index ready  (persist_dir={persist_dir})")
        return vs

    # ── load from disk ────────────────────────────────────────────────────

    @classmethod
    def load_faiss(
        cls,
        index_path: str,
        embeddings,
        docs: Optional[List[Document]] = None,
    ) -> "LegalVectorStore":
        """Load a previously saved FAISS index from disk."""
        from langchain_community.vectorstores import FAISS
        vs = FAISS.load_local(
            index_path, embeddings, allow_dangerous_deserialization=True
        )
        logger.info(f"✓ FAISS index loaded ← {index_path}")
        return cls(vs, docs or [])

    # ── search ────────────────────────────────────────────────────────────

    def search(
        self,
        query:      str,
        k:          int            = 5,
        law_id:     Optional[str]  = None,
        chunk_type: Optional[str]  = None,
    ) -> List[Document]:
        """
        Similarity search with optional metadata filters.

        Args:
            query:      Arabic query string.
            k:          Number of results to return.
            law_id:     Restrict to one law  (e.g. ``'penal_code'``).
            chunk_type: Restrict by type     (``'article'`` | ``'amended_article'``
                        | ``'preamble'`` | ``'table'``).
        """
        filter_dict: Dict[str, Any] = {}
        if law_id:
            filter_dict["law_id"]     = law_id
        if chunk_type:
            filter_dict["chunk_type"] = chunk_type

        if filter_dict:
            return self.vectorstore.similarity_search(query, k=k, filter=filter_dict)
        return self.vectorstore.similarity_search(query, k=k)

    def search_with_scores(
        self,
        query:  str,
        k:      int           = 5,
        law_id: Optional[str] = None,
    ) -> List[Tuple[Document, float]]:
        """Return ``(Document, score)`` pairs sorted by similarity."""
        filter_dict = {"law_id": law_id} if law_id else {}
        if filter_dict:
            return self.vectorstore.similarity_search_with_score(
                query, k=k, filter=filter_dict
            )
        return self.vectorstore.similarity_search_with_score(query, k=k)

    def search_latest(
        self,
        query:  str,
        k:      int           = 5,
        law_id: Optional[str] = None,
    ) -> List[Document]:
        """
        Search and return only the most recent version of each article
        among the hits. Useful when the index contains both original
        and amended chunks.
        """
        raw     = self.search(query, k=k * 3, law_id=law_id)
        deduped = filter_latest_versions(raw)
        return deduped[:k]

    def as_retriever(
        self,
        k:          int           = 5,
        law_id:     Optional[str] = None,
        chunk_type: Optional[str] = None,
    ):
        """Return a LangChain ``BaseRetriever`` for use in chains."""
        search_kwargs: Dict[str, Any] = {"k": k}
        filter_dict:   Dict[str, Any] = {}
        if law_id:
            filter_dict["law_id"]     = law_id
        if chunk_type:
            filter_dict["chunk_type"] = chunk_type
        if filter_dict:
            search_kwargs["filter"] = filter_dict
        return self.vectorstore.as_retriever(search_kwargs=search_kwargs)


print("✓ LegalVectorStore defined")


✓ LegalVectorStore defined


In [18]:
# =============================================================================
# CELL 4: CONVENIENCE ENTRY POINT
# =============================================================================

def build_vector_store(
    docs:               List[Document],
    embeddings,
    backend:            str           = "faiss",
    docs_filter:        str           = "latest",
    faiss_index_path:   Optional[str] = None,
    chroma_persist_dir: Optional[str] = "./chroma_legal_db",
    chroma_collection:  str           = "egyptian_legal_corpus",
    verbose:            bool          = True,
) -> LegalVectorStore:
    """
    One-call entry point to build a :class:`LegalVectorStore`.

    Args:
        docs:               Output of :func:`~src.Chunking.chunking.get_chunks`.
        embeddings:         LangChain ``Embeddings`` object.
        backend:            ``'faiss'`` | ``'chroma'``.
        docs_filter:        ``'latest'``   — amended version wins over original ✓
                            ``'all'``      — index everything
                            ``'original'`` — skip amendments entirely
        faiss_index_path:   Path to save FAISS index (``None`` = don't save).
        chroma_persist_dir: Chroma persistence directory.
        chroma_collection:  Chroma collection name.
        verbose:            Print progress.

    Returns:
        :class:`LegalVectorStore`
    """
    print("\n" + "=" * 70)
    print("BUILDING VECTOR STORE")
    print("=" * 70)
    print(f"  Input documents : {len(docs):,}")
    print(f"  Backend         : {backend}")
    print(f"  Filter          : {docs_filter}")

    return LegalVectorStore.from_documents(
        docs               = docs,
        embeddings         = embeddings,
        backend            = backend,
        docs_filter        = docs_filter,
        faiss_index_path   = faiss_index_path,
        chroma_persist_dir = chroma_persist_dir,
        chroma_collection  = chroma_collection,
        verbose            = verbose,
    )


print("✓ build_vector_store defined")

✓ build_vector_store defined


In [19]:
# =============================================================================
# CELL 5: RUN
# =============================================================================

chunks = get_chunks()

embeddings = HuggingFaceEmbeddings(model_name=get_settings().EMBEDDING_MODEL)

lvs = build_vector_store(
    docs             = chunks,
    embeddings       = embeddings,
    backend          = "faiss",
    docs_filter      = "latest",
    faiss_index_path = "legal_faiss_index",
)

print(f"\n✓ Vector store built — {len(lvs._docs):,} indexed documents")


2026-03-09 06:15:49,560 - INFO -   3okobat.txt [article] → 18 chunks
2026-03-09 06:15:49,561 - INFO -   new_3okobat_num36_2020.txt [amendment] → 1 chunks
2026-03-09 06:15:49,561 - INFO -   new_3okobat_num6_2020.txt [amendment] → 1 chunks
2026-03-09 06:15:49,561 - INFO -   قانون-مكافحة-غسل-الامول_ultra_clean.txt [article] → 1 chunks
2026-03-09 06:15:49,562 - INFO -   asle7a.txt [article] → 3 chunks
2026-03-09 06:15:49,562 - INFO -   asle7a_tables.txt [table] → 1 chunks
2026-03-09 06:15:49,563 - INFO -   dostor_masr.txt [article] → 9 chunks
2026-03-09 06:15:49,564 - INFO -   drugs.txt [article] → 3 chunks
2026-03-09 06:15:49,564 - INFO -   drugs_tables.txt [table] → 1 chunks
2026-03-09 06:15:49,565 - INFO -   egra2at_gena2ya.txt [article] → 16 chunks
2026-03-09 06:15:49,567 - INFO -   erhab.txt [article] → 4 chunks
2026-03-09 06:15:49,568 - INFO -   قانون-الطوارئ_ultra_clean.txt [article] → 1 chunks
2026-03-09 06:15:49,568 - INFO -   tech.txt [article] → 4 chunks
2026-03-09 06:15:49,569 


BUILDING VECTOR STORE
  Input documents : 63
  Backend         : faiss
  Filter          : latest
  Building FAISS index over 63 documents …


2026-03-09 06:15:57,334 - INFO -   ✓ FAISS index saved → legal_faiss_index


  ✓ FAISS index ready

✓ Vector store built — 63 indexed documents


In [ ]:
# =============================================================================
# CELL 6: QUICK SANITY CHECK
# =============================================================================

from IPython.display import Markdown, display

# Show the first indexed document
first = lvs._docs[0]
print("\nFirst indexed document metadata:")
for k, v in first.metadata.items():
    print(f"  {k}: {v}")

display(Markdown(first.page_content[:500]))